In [ ]:
import polars as pl
import numpy as np

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
df = pl.read_csv("../data/stockdata3.csv")

In [ ]:
stocks = [c for c in df.columns if c != "day" and c != "timestr"]
returns = [f"r{s}" for s in stocks]

In [ ]:
nums = [5, 10, 15, 20, 30, 60, 120, 180]
sfreqs = [f"{n}m" for n in nums]

In [ ]:
# !pip install scipy

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
df = (
    df.with_columns(
        (pl.date(2000, 1, 1) + pl.duration(days=pl.col("day") - 1))
        .dt.combine(pl.col("timestr").str.to_time("%H:%M:%S"))
        .alias("datetime")
    )
    if "datatime" not in df.columns
    else df
)
display(df.select("day", "timestr", "datetime"))
df = df.with_row_index() if "index" not in df.columns else df

In [ ]:
def draw_trading_time(df, cols):
    sdf = (
        df.group_by("day")
        .agg(
            # (pl.col("datetime").max() - pl.col("datetime").min()).alias("span")
            [pl.col(c).drop_nans().drop_nulls().count().alias(c) for c in cols]
        )
        .sort("day")
    )
    display(sdf.filter())
    for c in cols:
        plt.plot(sdf[c], label=c)
    plt.legend()
    plt.show()
    return sdf

In [ ]:
def daily_vol(df, s):
    return df.group_by("day").agg(pl.col(s).pow(2).sum().sqrt())

In [ ]:
INTRA = 0
EXTRA = 1


def get_ret(df, s):
    return (
        df.sort("datetime")
        .fill_nan(None)
        .select(
            [
                pl.col("day"),
                pl.col("datetime"),
                ((pl.col(s) / pl.col(s).shift(1)).log() * 1e4).alias(f"r{s}"),
            ]
        )
    )


def add_ret(df):
    return (
        df.sort("datetime")
        .fill_nan(None)
        .with_columns(
            [
                ((pl.col(s) / pl.col(s).shift(1)).log() * 1e4).alias(f"r{s}")
                for s in stocks
            ]
        )
        .with_columns(
            [
                pl.when(pl.col("day").diff() == 0)
                .then(pl.lit(INTRA))
                .otherwise(pl.lit(EXTRA))
                .alias("return_type")
            ]
        )
    )

In [ ]:
sdf = draw_trading_time(df, stocks)
sdf.filter(pl.col("f") < 100)

In [ ]:
df.filter(df["day"] == 8).filter(~pl.col("f").is_null())

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def draw_df(df=df, stocks=stocks, window=None):
    if window is not None:
        start, end = window.split("-")
        start = pl.Series([start]).str.to_datetime("%Y%m%d")[0]
        end = pl.Series([end]).str.to_datetime("%Y%m%d")[0]
        df = df.filter(pl.col("datetime").is_between(start, end))
    fig = make_subplots(
        rows=len(stocks),
        cols=2,
        shared_xaxes=True,
        column_widths=[0.72, 0.28],
        horizontal_spacing=0.07,
        vertical_spacing=0.04,
        subplot_titles=[
            title for s in stocks for title in (f"ts for {s}", f"hist for {s}")
        ],
    )

    datetime = df["datetime"].to_numpy()
    for row, s in enumerate(stocks, start=1):
        values = df[s].to_numpy()
        fig.add_trace(
            go.Scattergl(
                x=datetime, y=values, line_shape="hv", name=s, showlegend=False
            ),
            row=row,
            col=1,
        )
        # fig.add_trace(
        #     go.Histogram(x=values, nbinsx=100, name=s, showlegend=False),
        #     row=row,
        #     col=2,
        # )

    fig.update_layout(
        height=260 * len(stocks),
        title_text="Stock time series & distributions",
        margin=dict(t=60),
    )
    fig.show()

In [ ]:
draw_df()

In [ ]:
# plt.plot(df["day"])
df = df.with_columns(
    pl.when(pl.col("a") == 0).then(None).otherwise(pl.col("a")).alias("a"),
    pl.when(pl.col("d") == 1).then(None).otherwise(pl.col("d")).alias("d"),
)

In [ ]:
# draw_df(df)

In [ ]:
# deal with c: likely split
def normalize_c(df):
    idxs = df.filter(pl.col("c").diff().abs() > 0.4 * pl.col("c"))["index"]
    if len(idxs) > 0:
        index = idxs[0]
    else:
        return df
    print(index)
    return df.with_columns(
        pl.when(pl.col("index") < index)
        .then(pl.col("c") / 2)
        .otherwise(pl.col("c"))
        .alias("c")
    )

In [ ]:
# deal with c: likely split
def mask_bad(col, thd=100, neighbor=1):
    ratio = thd / 1e4  # bps
    ret = (pl.col(col) / pl.col(col).shift(1)).log()
    mask = False
    for n in range(1, neighbor + 1):
        mask |= (
            (ret * ret.shift(-n) < 0) & (ret.abs() > ratio) & (ret.shift(-n).abs() > ratio)
        )
    return mask


def mask_good(col, thd=100, neighbor=1):
    return ~mask_bad(col, thd, neighbor)


# def clean_d(df):
#     return df

In [ ]:
def downsample(df, freq):
    return df.group_by_dynamic("datetime", every=freq, group_by="day").agg(
        pl.all().last()
    )

In [ ]:
# df_5m = downsample(df, "5m")
# df_10m = downsample(df, "10m")
# df_15m = downsample(df, "15m")
dfs = {"1m": df} | {f: downsample(df, f) for f in sfreqs}

In [ ]:
# df.filter(pl.col("rd").abs() > 10).select("day", "timestr", "d", "rd")
# display(df.filter(mask_bad("d", 100)).select("index", "day", "timestr", "d", "rd"))
# display(df[:1756].select("index", "day", "timestr", "d", "rd"))

In [ ]:
from scipy import stats


def plot_one_qq(data, label=None, plot=plt, **kwargs):
    (osm, osr), (slope, intercept, r) = stats.probplot(data, plot=None)
    plot.plot(osm, osr, "o", label=label, **kwargs)
    plot.plot(osm, slope * osm + intercept, "r-")


def check_bad(df, c, thd=100):
    tot = len(get_ret(df, c)[f"r{c}"].drop_nulls())
    ret = get_ret(df.filter(~mask_bad(c, thd)), c)[f"r{c}"]
    plot_one_qq(ret, label=str(thd) + "," + str(len(ret) / tot), markersize=2)


def check_down(df, c, freq="5m"):
    tot = len(get_ret(df, c)[f"r{c}"].drop_nulls())
    ret = get_ret(downsample(df, freq), c)[f"r{c}"]
    plot_one_qq(ret, label=freq + "," + str(len(ret) / tot), markersize=2)

In [ ]:
for t in [10000, 1000, 500, 100, 50, 10, 5]:
    check_bad(df, "d", t)

plt.legend()
plt.show()

In [ ]:
for t in ["5m", "6m", "10m", "11m", "15m", "16m"]:
    check_down(df, "b", t)

plt.legend()
plt.show()

In [ ]:
for f in dfs:
    dfs[f] = normalize_c(dfs[f])

In [ ]:
# # df["day"].diff().value_counts()
# for f in dfs:
#     draw_df(dfs[f])

In [ ]:
# for s in stocks:
#     null_df_s = df.filter(pl.col(s).is_null())
#     display(null_df_s)
#     print(s, len(null_df_s))
# plt.hist(null_df_s['index'].diff())
import polars as pl


def add_null_blocks(df: pl.DataFrame, col: str) -> pl.DataFrame:
    is_null = pl.col(col).is_null()
    start = is_null & ~is_null.shift(1).over("day").fill_null(False)

    return (
        df.sort(["day", "timestr"])
        .with_columns(
            start.alias("_start"),
        )
        .with_columns(
            pl.when(is_null)
            .then(pl.col("_start").cum_sum().over("day"))
            .otherwise(None)
            .alias(f"{col}_null_block")
        )
        .with_columns(
            pl.when(is_null)
            .then(pl.len().over(["day", f"{col}_null_block"]))
            .otherwise(None)
            .alias(f"{col}_null_block_len")
        )
        .drop("_start")
    )

In [ ]:
def stock_day_null_quality(blocks, stock):
    return (
        blocks.group_by("day")
        .agg(
            pl.col("len").max().alias(f"{stock}_max_null_block"),
            pl.col("len").sum().alias(f"{stock}_null_minutes"),
        )
        .with_columns(
            (pl.col(f"{stock}_max_null_block") > 30).alias(f"{stock}_bad_null_day")
        )
    )

In [ ]:
for s in stocks:
    df = add_null_blocks(df, s)

In [ ]:
for s in stocks:
    blocks = (
        df.filter(pl.col(s).is_null())
        .group_by(["day", f"{s}_null_block"])
        .agg(
            pl.col("timestr").first().alias("start"),
            pl.col("timestr").last().alias("end"),
            pl.len().alias("len"),
        )
    ).sort("day")
    q = stock_day_null_quality(blocks, s)
    print(f"=========================={s}=======================")
    display(blocks)
    print(blocks.sort("len")["len"].value_counts())
    display(q)

In [ ]:
for f in dfs:
    dfs[f] = add_ret(dfs[f])

In [ ]:
for s in stocks:
    print(f"=============={s}=============")
    for f in dfs:
        vol = daily_vol(dfs[f], f"r{s}").sort("day")
        plt.step(vol["day"], vol[f"r{s}"], label=f)
    plt.legend()
    plt.show()

In [ ]:
lines = []


for t in [10000, 1000, 500, 100, 50, 10]:
    print(f"filtered by {t}")
    tdf = dfs["1m"].filter(mask_good("d", t, 2))
    for f in dfs:
        vol = daily_vol(downsample(tdf, f), f"rd").sort("day")
        plt.step(vol["day"], vol[f"rd"], label=f)
    # vol = daily_vol(get_ret(tdf, "d"), "rd").sorfst("day")
    # plt.step(vol["day"], vol[f"rd"], label=t, ls=":")
    plt.legend()
    plt.show()

In [ ]:
df.describe()

In [ ]:
# !pip install statsmodels

In [ ]:
# df.select(returns).describe()

In [ ]:
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf


def summarize_returns(df, stocks=stocks, log=False, draw_trace=False):
    returns = [f"r{s}" for s in stocks]
    for s, r in zip(stocks, returns):
        print(f"=========================={r}=======================")
        print(df[r].describe())
        print(f"zero_pct = {(df[r] == 0).mean() * 100} %")
        plt.hist(df[r], bins=200, log=log)
        plt.show()
        stats.probplot((df[r] - df[r].mean()) / df[r].std(), plot=plt)
        # lims = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]
        lims = [-4, 4]
        plt.plot(lims, lims, "r--")
        plt.show()
        plot_acf(df[r].drop_nulls().to_numpy(), lags=40, alpha=0.05)
        plt.show()
        if draw_trace:
            draw_df(df, [s])

In [ ]:
# summarize_returns(df.filter(pl.col("return_type") == EXTRA))
# summarize_returns(df)
# for f, df in dfs.items():
#     print(f"!!!! {f}")
#     summarize_returns(df.filter(pl.col("return_type") == INTRA), stocks=["b"], log=True)

In [ ]:
# for f, df in dfs.items():
#     print(f"!!!! {f}")
#     summarize_returns(
#         df.filter(pl.col("return_type") == INTRA).filter(pl.col("rd").abs() < 100),
#         stocks=["d"],
#         log=True,
#     )

In [ ]:
for f in dfs:
    print(f"========={f}==========")
    summarize_returns(
        downsample(dfs["1m"].filter(pl.col("return_type") == INTRA).filter(mask_good('d', 100)), f),
        stocks=["d"],
        log=True,
    )

In [ ]:
summarize_returns(
    dfs["15m"].filter(pl.col("return_type") == INTRA),
    stocks=stocks,
    log=True,
)

In [ ]:
# draw_df(df, ["d"])
# draw_df(df, ["rd"])
summarize_returns(
    dfs["120m"].filter(pl.col("return_type") == INTRA),
    stocks=stocks,
    log=True,
)

In [ ]:
window='20000702-20000704'
draw_df(dfs['120m'], ['b'], window=window)
draw_df(dfs['60m'], ['b'], window=window)
draw_df(dfs['30m'], ['b'], window=window)
draw_df(dfs['15m'], ['b'], window=window)
draw_df(dfs['5m'], ['b'], window=window)
draw_df(dfs['1m'], ['b'], window=window)
# dfs['1m']

In [ ]:
window='20000702-20000704'
# draw_df(dfs['60m'], ['d'], window=window)
# draw_df(dfs['30m'], ['d'], window=window)
# draw_df(dfs['15m'], ['d'], window=window)
# draw_df(dfs['10m'], ['d'], window=window)
# draw_df(dfs['5m'], ['d'], window=window)
# draw_df(dfs['1m'], ['d'], window=window)
neighbor = 2
draw_df(dfs['1m'].filter(mask_good('d', 10000, neighbor)), ['d'], window=window)
draw_df(dfs['1m'].filter(mask_good('d', 1000, neighbor)), ['d'], window=window)
draw_df(dfs['1m'].filter(mask_good('d', 500, neighbor)), ['d'], window=window)
draw_df(dfs['1m'].filter(mask_good('d', 50, neighbor)), ['d'], window=window)
draw_df(dfs['1m'].filter(mask_good('d', 10, neighbor)), ['d'], window=window)
# draw_df(dfs['1m'].select([pl.col('datetime'), mask_good('d', 10).alias('d')]), ['d'], window=window)

In [ ]:
# for f, check Poisson-likeness

Nd = (
    dfs["1m"]
    .group_by("day")
    .agg(((pl.col("f").diff() != 0) & (pl.col("f").is_not_null())).sum())
)
jumps = dfs["1m"].filter(pl.col("rf") != 0).select(["day", "rf"])

In [ ]:
Nd.std() / Nd.mean()

In [ ]:
plt.hist(jumps["rf"], bins=200)
plt.show()
stats.probplot(
    (jumps["rf"] - jumps["rf"].mean()) / jumps["rf"].std(), plot=plt, dist=stats.t(df=4)
)
plt.show()

In [ ]:
df.filter(pl.col("rf").abs() > 0)["rf"].describe()

In [ ]:
for r in returns:
    print(r, df.filter(pl.col(r) > 0)[r].abs().min())